# Olist E-Commerce ELT Pipeline: Bronze to Silver Layer
Notebook này thực hiện tiến trình trích xuất và làm sạch dữ liệu từ tầng **Bronze** (dữ liệu thô - Raw CSV) để chuyển đổi sang tầng **Silver** (dữ liệu đã được chuẩn hoá cấu trúc và làm sạch cơ bản) dưới định dạng **Delta Lake**.

### Mục tiêu xử lý:
1. Thiết lập kết nối bảo mật tới Azure ADLS Gen2 Storage Account.
2. Kiểm tra tổng quan hệ thống dữ liệu: Đếm số dòng, phát hiện dữ liệu lặp (Duplicates) và định vị các cột bị khuyết thiếu (Null/Empty).
3. Chuẩn hoá kiểu dữ liệu (Data Types): Ép chuỗi thời gian thành Timestamp, đồng bộ hóa văn bản (Lower/Trim) để tránh lỗi phân nhóm.
4. Loại bỏ các bản ghi tài chính lỗi (Doanh thu/Phí ship bị âm).
5. Tiêm siêu dữ liệu thực thi (`processed_at`) để phục vụ kiểm toán Data Lineage.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, trim, when, current_timestamp, datediff, to_timestamp, expr, coalesce, lit 

# 1. KẾT NỐI ĐẾN STORAGE ACCOUNT
storage_account = "mystorageaccount"
storage_key = "my-hashkey"
spark.conf.set(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key)

# 2. TẠO ĐƯỜNG DẪN
bronze_path = "abfss://bronze@{storage_account}.dfs.core.windows.net/"
silver_path = "abfss://silver@{storage_account}.dfs.core.windows.net/"


# 3. ĐỌC FOLDER BRONZE
df_orders = spark.read.csv(
    bronze_path + "olist_orders_dataset/",
    header=True,
    inferSchema=True
)
df_products = spark.read.csv(
    bronze_path + "olist_products_dataset/",
    header=True,
    inferSchema=True
)

df_sellers = spark.read.csv(
    bronze_path + "olist_sellers_dataset/",
    header=True,
    inferSchema=True
)

df_customers = spark.read.csv(
    bronze_path + "olist_customers_dataset/",
    header=True,
    inferSchema=True
)

df_geolocation = spark.read.csv(
    bronze_path + "olist_geolocation_dataset/",
    header=True,
    inferSchema=True
)

df_order_items = spark.read.csv(
    bronze_path + "olist_order_items_dataset/",
    header=True,
    inferSchema=True
)

df_order_payments = spark.read.csv(
    bronze_path + "olist_order_payments_dataset/",
    header=True,
    inferSchema=True
)

df_order_reviews = spark.read.csv(
    bronze_path + "olist_order_reviews_dataset/",
    header=True,
    inferSchema=True,
    multiLine=True,
    escape='"'
)
# HIỂN THỊ MỘT VÀI DỮ LIỆU KIỂM TRA
display(df_orders.limit(5))
display(df_products.limit(5))
display(df_sellers.limit(5))
display(df_customers.limit(5))
display(df_geolocation.limit(5))
display(df_order_items.limit(5))
display(df_order_payments.limit(5))
display(df_order_reviews.limit(5))

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33Z,2017-10-02T11:07:15Z,2017-10-04T19:55:00Z,2017-10-10T21:25:13Z,2017-10-18T00:00:00Z
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37Z,2018-07-26T03:24:27Z,2018-07-26T14:31:00Z,2018-08-07T15:27:45Z,2018-08-13T00:00:00Z
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08T08:38:49Z,2018-08-08T08:55:23Z,2018-08-08T13:50:00Z,2018-08-17T18:06:29Z,2018-09-04T00:00:00Z
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18T19:28:06Z,2017-11-18T19:45:59Z,2017-11-22T13:39:59Z,2017-12-02T00:28:42Z,2017-12-15T00:00:00Z
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13T21:18:39Z,2018-02-13T22:20:29Z,2018-02-14T19:46:34Z,2018-02-16T18:17:02Z,2018-02-26T00:00:00Z


product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14
3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20
96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154,18,9,15
cef67bcfe19066a932b7673e239eb23d,bebes,27,261,1,371,26,4,26
9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37,402,4,625,20,17,13


seller_id,seller_zip_code_prefix,seller_city,seller_state
3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
1037,-23.54562128115268,-46.63929204800168,sao paulo,SP
1046,-23.546081127035535,-46.64482029837157,sao paulo,SP
1046,-23.54612896641469,-46.64295148361138,sao paulo,SP
1041,-23.5443921648681,-46.63949930627844,sao paulo,SP
1035,-23.541577961711493,-46.64160722329613,sao paulo,SP


order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19T09:45:35Z,58.9,13.29
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03T11:05:13Z,239.9,19.93
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18T14:48:30Z,199.0,17.87
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15T10:10:18Z,12.99,12.79
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13T13:57:51Z,199.9,18.14


order_id,payment_sequential,payment_type,payment_installments,payment_value
b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,null,null,2018-01-18T00:00:00Z,2018-01-18T21:46:59Z
80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,null,null,2018-03-10T00:00:00Z,2018-03-11T03:05:13Z
228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,null,null,2018-02-17T00:00:00Z,2018-02-18T14:36:24Z
e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,null,Recebi bem antes do prazo estipulado.,2017-04-21T00:00:00Z,2017-04-21T22:02:06Z
f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,null,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,2018-03-01T00:00:00Z,2018-03-02T10:26:53Z


## Data Quality Audit (Kiểm tra chất lượng dữ liệu)
Tiến hành quét qua toàn bộ 8 bảng dữ liệu thô để phát hiện các lỗ hổng dữ liệu rác, dòng trùng lặp và phân tích sơ bộ tình trạng khuyết thiếu trước khi áp dụng các hàm Transformation logic.

In [ ]:
tables = {
    "customers": df_customers,
    "geolocation": df_geolocation,
    "order_items": df_order_items,
    "order_payments": df_order_payments,
    "order_reviews": df_order_reviews,
    "orders": df_orders,
    "products": df_products,
    "sellers": df_sellers
}

# TỔNG SỐ DÒNG
total_rows_dataset = sum([df.count() for df in tables.values()])
print(f"Tổng số dòng trong dataset: {total_rows_dataset}")

In [ ]:
# KIỂM TRA NULL, SỐ DÒNG VÀ CẤU TRÚC TỪNG BẢNG
from pyspark.sql import functions as F
from pyspark.sql.types import NumericType, StringType

def count_null_columns(df):
    null_counts = []
    for field in df.schema.fields:
        col_name = field.name
        data_type = field.dataType

        condition = df[col_name].isNull()

        if isinstance(data_type, StringType):
            condition = condition | (df[col_name] == "")
        elif isinstance(data_type, NumericType):
            condition = condition | F.isnan(df[col_name])

        null_count = df.filter(condition).count()
        if null_count > 0:
            null_counts.append(f"   + Cột '{col_name}': {null_count:,} dòng trống")
    return null_counts

for name, df in tables.items():
    print(f"\nBẢNG: {name.upper()}")

    row_count = df.count()
    print(f" 🔹 Tổng số dòng thực tế: {row_count:,}")

    distinct_count = df.distinct().count()
    duplicate_count = row_count - distinct_count
    print(f" 🔹 Số lượng dòng bị lặp: {duplicate_count:,}")
    if duplicate_count > 0:
        print("Phát hiện dữ liệu lặp!")
    null_info = count_null_columns(df)
    if null_info:
        print(f" 🔹 Tình trạng dữ liệu (Null / Empty):")
        for info in null_info:
            print(info)
    else:
        print(" 🔹 Tình trạng dữ liệu: Sạch (0 dòng trống)")
    df.printSchema()
    print("-" * 56)

In [ ]:
# KIỂM TRA CHI TIẾT DOANH THU VÀ CHI PHÍ TRÊN BẢNG ORDER_ITEMS
display(df_order_items.select("price", "freight_value").describe())

negative_price = df_order_items.filter(F.col("price") < 0).count()
negative_freight = df_order_items.filter(F.col("freight_value") < 0).count()

print(f"Số lượng dòng có Doanh thu (price) bị âm      : {negative_price}")
print(f"Số lượng dòng có Phí vận chuyển (freight) bị âm: {negative_freight}")

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
def add_pipeline_metadata(df):
    """
    Helper function tiêm siêu dữ liệu hệ thống (Lineage) vào dòng dữ liệu.
    Giúp theo dõi thời điểm pipeline thực thi xử lý bản ghi tại RAM máy ảo Colab.
    """
    return df.withColumn("processed_at", F.current_timestamp())

def clean_text_column(col_name):
    """
    Helper function đồng bộ hoá chuỗi phân loại phân khúc văn bản:
    Khử khoảng trắng thừa (trim) và chuyển về chữ thường viết thường (lower).
    Chặn đứng lỗi Power BI phân tách nhầm nhóm do chữ hoa/chữ thường.
    """
    return F.lower(F.trim(F.col(col_name)))

## Data Transformation & Standardization (Chuẩn hoá & Mục đích)

Phần này thực hiện áp dụng các hàm bổ trợ (Helper Functions) đã định nghĩa ở trên để tiến hành biến đổi dữ liệu đồng bộ cho toàn bộ 8 bảng thành phần trước khi đưa vào kho lưu trữ Silver.

### 🎯 Mục đích cốt lõi:
* **Đồng bộ hóa chuỗi văn bản (`clean_text_column`):** Xử lý triệt để các trường dữ liệu định danh như tên thành phố (`customer_city`, `seller_city`) và mã tiểu bang (`customer_state`, `seller_state`). Việc khử khoảng trắng thừa (Trim) và chuyển về chữ thường (Lower) giúp ngăn chặn hoàn toàn lỗi Power BI phân tách nhầm nhóm do ký tự hoa/thường, đảm bảo bối cảnh bộ lọc (Filter Context) trên giao diện Dashboard hoạt động chính xác 100%.
* **Kiểm toán vòng đời dữ liệu (`add_pipeline_metadata`):** Tự động tiêm thêm thuộc tính siêu dữ liệu hệ thống `processed_at` (lấy theo thời gian thực thi của RAM máy ảo). Điều này giúp xây dựng tính năng Data Lineage, hỗ trợ đội ngũ Kỹ sư dữ liệu dễ dàng kiểm toán, tra cứu và xác định chính xác thời điểm bản ghi được hệ sinh thái Big Data xử lý.

In [ ]:
# ORDERS
df_orders_silver = df_orders.select(
    F.col("order_id"),
    F.col("customer_id"),
    clean_text_column("order_status").alias("order_status"),
    F.to_timestamp(F.col("order_purchase_timestamp"), "yyyy-MM-dd HH:mm:ss").alias("order_purchase_timestamp"),
    F.to_timestamp(F.col("order_approved_at"), "yyyy-MM-dd HH:mm:ss").alias("order_approved_at"),
    F.to_timestamp(F.col("order_delivered_carrier_date"), "yyyy-MM-dd HH:mm:ss").alias("order_delivered_carrier_date"),
    F.to_timestamp(F.col("order_delivered_customer_date"), "yyyy-MM-dd HH:mm:ss").alias("order_delivered_customer_date"),
    F.to_timestamp(F.col("order_estimated_delivery_date"), "yyyy-MM-dd HH:mm:ss").alias("order_estimated_delivery_date")
).withColumn(
    "delivery_days",
    F.datediff(F.col("order_delivered_customer_date"), F.col("order_purchase_timestamp"))
).withColumn(
    "is_valid_order",
    (F.col("order_id").isNotNull()) &
    (F.col("order_purchase_timestamp").isNotNull()) &
    (F.coalesce(F.col("delivery_days") >= 0, F.lit(True))) # Nếu chưa giao (Null) vẫn coi là True tạm thời
).na.fill({"order_status": "unknown"})\
 .dropDuplicates(["order_id"])

df_orders_silver = add_pipeline_metadata(df_orders_silver)

# PRODUCTS
df_products_silver = df_products.select(
    F.col("product_id"),
    F.when(F.col("product_category_name").isNull(), F.lit("unknown"))
     .otherwise(clean_text_column("product_category_name")).alias("product_category_name"),
    F.col("product_name_lenght"),
    F.col("product_description_lenght"),
    F.col("product_photos_qty"),
    F.col("product_weight_g"),
    F.col("product_length_cm"),
    F.col("product_height_cm"),
    F.col("product_width_cm")
).dropDuplicates(["product_id"])

df_products_silver = add_pipeline_metadata(df_products_silver)

# SELLERS
df_sellers_silver = df_sellers.select(
    F.col("seller_id"),
    F.col("seller_zip_code_prefix"),
    clean_text_column("seller_city").alias("seller_city"),
    clean_text_column("seller_state").alias("seller_state")
).dropDuplicates(["seller_id"])

df_sellers_silver = add_pipeline_metadata(df_sellers_silver)

# CUSTOMERS
df_customers_silver = df_customers.select(
    F.col("customer_id"),
    F.col("customer_unique_id"),
    F.col("customer_zip_code_prefix"),
    clean_text_column("customer_city").alias("customer_city"),
    clean_text_column("customer_state").alias("customer_state")
).dropDuplicates(["customer_id"])

df_customers_silver = add_pipeline_metadata(df_customers_silver)

# GEOLOCATION
window_mode = Window.partitionBy("geolocation_zip_code_prefix").orderBy(F.col("frequency_count").desc(), F.col("geolocation_city").asc())

df_geo_counted = df_geolocation.select(
    F.col("geolocation_zip_code_prefix"),
    F.col("geolocation_lat").cast("double"),
    F.col("geolocation_lng").cast("double"),
    clean_text_column("geolocation_city").alias("geolocation_city"),
    clean_text_column("geolocation_state").alias("geolocation_state")
).groupBy("geolocation_zip_code_prefix", "geolocation_city", "geolocation_state") \
 .agg(
    F.count("*").alias("frequency_count"),
    F.avg("geolocation_lat").alias("avg_lat"),
    F.avg("geolocation_lng").alias("avg_lng")
 )

df_geolocation_silver = df_geo_counted.withColumn("rank", F.row_number().over(window_mode)) \
    .filter(F.col("rank") == 1) \
    .select(
        F.col("geolocation_zip_code_prefix"),
        F.col("avg_lat").alias("geolocation_lat"),
        F.col("avg_lng").alias("geolocation_lng"),
        F.col("geolocation_city"),
        F.col("geolocation_state")
    )

df_geolocation_silver = add_pipeline_metadata(df_geolocation_silver)

# ORDER ITEMS
df_order_items_silver = df_order_items.select(
    F.col("order_id"),
    F.col("order_item_id"),
    F.col("product_id"),
    F.col("seller_id"),
    F.col("shipping_limit_date").cast("timestamp"),
    F.col("price").cast("double"),
    F.col("freight_value").cast("double")
).withColumn(
    "is_valid_financial",
    (F.col("order_id").isNotNull()) & (F.col("price") >= 0) & (F.col("freight_value") >= 0)
).dropDuplicates(["order_id", "order_item_id"])

df_order_items_silver = add_pipeline_metadata(df_order_items_silver)

# ORDER PAYMENTS
df_order_payments_silver = df_order_payments.select(
    F.col("order_id"),
    F.col("payment_sequential").cast("int"),
    clean_text_column("payment_type").alias("payment_type"),
    F.col("payment_installments").cast("int"),
    F.col("payment_value").cast("double")
).withColumn(
    "is_valid_payment",
    F.col("payment_value") >= 0
)

df_order_payments_silver = add_pipeline_metadata(df_order_payments_silver)

# ORDER REVIEWS
df_order_reviews_silver = df_order_reviews.select(
    F.col("review_id"),
    F.col("order_id"),
    F.expr("TRY_CAST(review_score AS INT)").alias("review_score"),
    F.trim(F.col("review_comment_title")).alias("review_comment_title"),
    F.trim(F.col("review_comment_message")).alias("review_comment_message")
).withColumn(
    "review_comment_title",
    F.when((F.col("review_comment_title") == "") | (F.col("review_comment_title").isNull()), F.lit("no title"))
     .otherwise(F.col("review_comment_title"))
).withColumn(
    "review_comment_message",
    F.when((F.col("review_comment_message") == "") | (F.col("review_comment_message").isNull()), F.lit("no message"))
     .otherwise(F.col("review_comment_message"))
).dropDuplicates(["review_id"])

df_order_reviews_silver = add_pipeline_metadata(df_order_reviews_silver)

## Delta Lake Storage Ingestion
Thực hiện ghi đè dữ liệu (Overwrite Mode) đã làm sạch vào phân vùng hệ thống `Silver` dưới định dạng Delta Lake, kích hoạt thuộc tính `overwriteSchema` để bảo toàn tính linh hoạt của cấu trúc bảng.

In [0]:
# 5. LƯU VÀO FOLDER SILVER DƯỚI DẠNG PARQUET FILE
print("✅ Đang lưu trữ dữ liệu vào Delta Lake...")
df_orders_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_path + "orders")

df_products_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_path + "products")

df_sellers_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_path + "sellers")

df_customers_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_path + "customers")

df_geolocation_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_path + "geolocation")

df_order_items_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_path + "order_items")

df_order_payments_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_path + "order_payments")

df_order_reviews_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_path + "order_reviews")

print("✅ Hoàn thành xử lý và lưu trữ toàn bộ 8 bảng tầng Silver!")



✅ Đang lưu trữ dữ liệu vào Delta Lake...
✅ Hoàn thành xử lý và lưu trữ toàn bộ 8 bảng tầng Silver!


In [0]:
# XEM DỮ LIỆU Ở TẦNG SILVER
silver_df = spark.read.format("delta").load(silver_path + "order_items")
display(silver_df)

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,processed_at
0039500db36d50d7d0bfaf23723550f3,1,62984ea1bba7fcea1f5b57084d3bf885,c3cfdc648177fdbbbb35635a37472c53,2018-07-10T10:51:09Z,169.9,33.33,2026-06-16T12:33:07.578617Z
00b44ba3d7c4a5e9a9ebafef9150781d,1,d9a5a5120d4d357871acd5408cb10e18,20d83f3ef0e6925fd74bfd59170babf7,2018-08-30T21:25:15Z,59.9,7.68,2026-06-16T12:33:07.578617Z
00c763284c0056eed753352f5559ff0a,1,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,2018-06-11T00:10:44Z,79.9,14.65,2026-06-16T12:33:07.578617Z
01405161cfdf60dd80df1f393a2796d5,1,f2bcc887769f563eb4f126389e64715a,4c8b8048e33af2bf94f2eb547746a916,2018-08-15T21:30:14Z,59.9,14.73,2026-06-16T12:33:07.578617Z
01ba82170ebb683c73793bb809bbf1b9,3,d6fe3b4ddecd4a8393c6a1385de3bfb6,7c67e1448b00f6e969d365cea6b010ab,2018-02-20T15:11:35Z,192.97,33.59,2026-06-16T12:33:07.578617Z
01d0f6cd01d79574621a56a1b9e21019,1,266f0c6925e7873790f11b920c701190,128639473a139ac0f3e5f5ade55873a5,2017-09-28T19:35:08Z,17.9,16.79,2026-06-16T12:33:07.578617Z
0216ea85f929ce4795f933573c6bb055,1,db123b35396aafabb2d5b476557abed5,4a3ca9315b744ce9f8e9374361493884,2018-06-12T20:59:11Z,76.9,14.63,2026-06-16T12:33:07.578617Z
02db3eec8ec9d3a35e2afe15367f21b8,1,7c1bd920dbdf22470b68bde975dd3ccf,cc419e0650a3c5ba77189a1882b7556a,2018-04-03T03:50:55Z,58.99,13.49,2026-06-16T12:33:07.578617Z
03cf8d977a18ea1308ec10c0f7de2f4e,1,d696750e550fd0f733979dd7e5dff921,9de4643a8dbde634fe55621059d92273,2017-04-26T21:10:10Z,169.99,16.4,2026-06-16T12:33:07.578617Z
03f9705f9135ccb809e74e902f19b2e8,1,cc68bcd2a2720e95aef8b6cb71f62ed9,a17f621c590ea0fab3d5d883e1630ec6,2017-05-02T21:25:13Z,19.33,10.96,2026-06-16T12:33:07.578617Z


In [0]:
# catalog_name = "olistpreprogressing_7405605323508273"
catalog_name = "hive_metastore"
silver_schema = "silver"

spark.sql(f"USE CATALOG `{catalog_name}`")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{silver_schema}`")

print(f"✅ Đã cấu hình thành công schema '{silver_schema}' trong catalog '{catalog_name}'.")

✅ Đã cấu hình thành công schema 'silver' trong catalog 'hive_metastore'.


In [0]:
# Ghi dữ liệu vào schema silver
def save_silver_table(df, table_name):
    """
    Hàm hỗ trợ ghi DataFrame Delta dưới dạng Managed Table vào Unity Catalog
    """
    full_table_name = f"{catalog_name}.{silver_schema}.{table_name}"
    print(f"💾 Đang lưu bảng: {full_table_name}...")
    
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_table_name)
    )
    print(f"✨ Đã ghi thành công bảng: {full_table_name}")

save_silver_table(df_customers_silver, "customers")

save_silver_table(df_geolocation_silver, "geolocation")

save_silver_table(df_order_items_silver, "order_items")

save_silver_table(df_order_payments_silver, "order_payments")

save_silver_table(df_order_reviews_silver, "order_reviews")

save_silver_table(df_orders_silver, "orders")

save_silver_table(df_products_silver, "products")

save_silver_table(df_sellers_silver, "sellers")

💾 Đang lưu bảng: olistpreprogressing_7405605323508273.silver.customers...
✨ Đã ghi thành công bảng: olistpreprogressing_7405605323508273.silver.customers
💾 Đang lưu bảng: olistpreprogressing_7405605323508273.silver.geolocation...
✨ Đã ghi thành công bảng: olistpreprogressing_7405605323508273.silver.geolocation
💾 Đang lưu bảng: olistpreprogressing_7405605323508273.silver.order_items...
✨ Đã ghi thành công bảng: olistpreprogressing_7405605323508273.silver.order_items
💾 Đang lưu bảng: olistpreprogressing_7405605323508273.silver.order_payments...
✨ Đã ghi thành công bảng: olistpreprogressing_7405605323508273.silver.order_payments
💾 Đang lưu bảng: olistpreprogressing_7405605323508273.silver.order_reviews...
✨ Đã ghi thành công bảng: olistpreprogressing_7405605323508273.silver.order_reviews
💾 Đang lưu bảng: olistpreprogressing_7405605323508273.silver.orders...
✨ Đã ghi thành công bảng: olistpreprogressing_7405605323508273.silver.orders
💾 Đang lưu bảng: olistpreprogressing_7405605323508273.si